# PO-Tool Backend Demo

Interactive notebook for testing all backend flows against Databricks.

## Prerequisites

1. Set environment variables: `DATABRICKS_HOST`, `DATABRICKS_TOKEN`, `DATABRICKS_WAREHOUSE_ID`, `DATABRICKS_CATALOG`, `DATABRICKS_SCHEMA`
2. Upload mock CSVs to Databricks (see `mock_data/DATA_DICTIONARY.md`)
3. Run cells 1 and 1b below to connect and create the app-managed tables

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("."))

import pandas as pd
from config.settings import DATABRICKS_HOST, DATABRICKS_WAREHOUSE_ID, DATABRICKS_TOKEN, DATABRICKS_CATALOG, DATABRICKS_SCHEMA
from po_tool.core.databricks import DatabricksClient

client = DatabricksClient(
    host=DATABRICKS_HOST,
    warehouse_id=DATABRICKS_WAREHOUSE_ID,
    token=DATABRICKS_TOKEN,
)

print(f"Host:      {DATABRICKS_HOST}")
print(f"Warehouse: {DATABRICKS_WAREHOUSE_ID}")
print(f"Catalog:   {DATABRICKS_CATALOG}")
print(f"Schema:    {DATABRICKS_SCHEMA}")
print("Client ready.")

## 1b. Create App Tables (run once)

Reads `schema.sql` and executes each CREATE TABLE statement against your Databricks warehouse.
Uses your catalog/schema from the env vars. Skip this cell if the tables already exist.

In [ ]:
from config.settings import qualified_table

with open("schema.sql") as f:
    schema_sql = f.read()

# Prefix each table name with catalog.schema
for table in ("query_queue", "query_results", "audit_log"):
    schema_sql = schema_sql.replace(f"IF NOT EXISTS {table}", f"IF NOT EXISTS {qualified_table(table)}")

statements = [s.strip() for s in schema_sql.split(";") if s.strip()]

for stmt in statements:
    if stmt.startswith("--"):
        continue
    print(f"Executing: {stmt[:60]}...")
    client.execute_sync(stmt)
    print("  Done.")

print("\nAll app tables created.")

## 2. e-Transfer: Single Search

Edit the `search_content` dict below and run the cell to submit a search.

In [ ]:
from po_tool.etransfer.queries import run_single_search

# --- Edit these parameters ---
username = "demo_user"
search_content = {
    "Search": ["Sender"],
    "EmailAddress": "john@example.com",
    "From": "2023-01-01",
    "To": "2025-12-31",
}
# ---

queue_id = run_single_search(client, username, search_content)
print(f"Search complete. Queue ID: {queue_id}")

## 3. e-Transfer: View Results

Fetch and display results for the queue ID from the previous cell.

In [ ]:
from po_tool.etransfer.queries import build_download_sql, ALWAYS_FIELDS
from config.settings import qualified_table

# --- Edit queue_id if needed ---
# queue_id = 1
# ---

sql = build_download_sql(f"QUEUE_ID = {queue_id}")
rows = client.execute_sync(sql)
df = pd.DataFrame(rows)
print(f"{len(df)} results")
df.head(20)

## 4. e-Transfer: Export to Excel

Select fields and export to an Excel file.

In [ ]:
from po_tool.etransfer.export import export_etransfer_to_file

# --- Edit selected_fields (None = all fields) ---
selected_fields = ["SENDER_EMAIL", "SENDER_NAME", "RECIPIENT_EMAIL", "TRANSFER_TYPE", "FRAUDULENT"]
# ---

path = export_etransfer_to_file(client, queue_id, f"output/etransfer_{queue_id}.xlsx", selected_fields)
print(f"Exported to: {path}")

## 5. Debit: Terminal Lookup

Search for a terminal by ID.

In [ ]:
from po_tool.debit.queries import build_terminal_query
from po_tool.core.audit import log_action

# --- Edit terminal_id ---
terminal_id = "T00000001"
# ---

sql = build_terminal_query(terminal_id)
rows = client.execute_sync(sql)
log_action(client, "demo_user", "Debit Terminal Search", {"terminal_id": terminal_id})

df = pd.DataFrame(rows)
print(f"{len(df)} terminals found")
df

## 6. Debit: Merchant Search

Search by merchant name, address, city, or postal code.

In [ ]:
from po_tool.debit.queries import build_merchant_query

# --- Edit filters (set to None to skip) ---
merchant_name = "Inc"
city = None
postal_code = None
# ---

sql = build_merchant_query(merchant_name=merchant_name, city=city, postal_code=postal_code)
rows = client.execute_sync(sql)
log_action(client, "demo_user", "Debit Merchant Search", {"merchant_name": merchant_name})

df = pd.DataFrame(rows)
print(f"{len(df)} merchants found")
df.head(20)

## 7. Debit: Transaction Search + Export

Search transactions for selected terminals and export to Excel.

In [ ]:
from po_tool.debit.queries import build_transaction_query
from po_tool.debit.export import export_debit_to_file

# --- Edit parameters ---
terminals = [("T00000001", "M000001")]   # (terminal_id, merchant_number)
from_date = "2021-01-01"
to_date = "2025-12-31"
amount = None    # optional exact amount
pan = None       # optional PAN pattern
# ---

sql = build_transaction_query(terminals, from_date, to_date, amount=amount, pan=pan)

# Transactions can be large — use async + EXTERNAL_LINKS
statement_id = client.submit_async(sql)
client.poll_until_done(statement_id)
rows = client.fetch_results_as_dicts(statement_id)

log_action(client, "demo_user", "Debit Transaction Search", {
    "terminals": [t[0] for t in terminals], "from": from_date, "to": to_date
})

df = pd.DataFrame(rows)
print(f"{len(df)} transactions found")
display(df.head(20))

if len(df) > 0:
    path = export_debit_to_file(rows, "output/debit_transactions.xlsx")
    print(f"Exported to: {path}")

## 8. Queue Overview

List all queue rows.

In [ ]:
from po_tool.core.queue import get_queue_rows

rows = get_queue_rows(client)
df = pd.DataFrame(rows)
print(f"{len(df)} queue rows")
df

## 9. Audit Log

View recent audit entries.

In [ ]:
from config.settings import qualified_table

sql = f"SELECT * FROM {qualified_table('audit_log')} ORDER BY TIMESTAMP DESC LIMIT 50"
rows = client.execute_sync(sql)
df = pd.DataFrame(rows)
print(f"{len(df)} audit entries")
df